<a href="https://colab.research.google.com/github/arelkeselbri/pgc305/blob/main/aula0_1_regressao_logistica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PGC305 - Tópicos especiais em LLM e Deep Learning

## Definição dos dados

In [ ]:
import torch; import sklearn

# 1. Carregar dados
iris = sklearn.datasets.load_iris()
X = iris.data        # 4 features: sépalas e pétalas
y = (iris.target == 1).astype(float)  # 1 se Versicolor, 0 caso contrário

# 2. Preparar dados para pytorch
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

## Definição do modelo e treinamento

In [ ]:
# 3. Definir modelo: regressão logística
modelo = torch.nn.Linear(4, 1)  # 4 features → 1 saída (probabilidade de ser Versicolor)

# 4. Definir função de perda e algoritmo de otimização
funcao_perda = torch.nn.BCEWithLogitsLoss()  # combinação de sigmoid + BCE
optimizer = torch.optim.SGD(modelo.parameters(), lr=0.1)

## Execução do treinamento

In [ ]:
# 5. Treino
for epoch in range(1000):
    optimizer.zero_grad() # reseta gradiente senão acumula
    outputs = modelo(X)
    loss = funcao_perda(outputs, y)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Época [{epoch+1}/100], Loss: {loss.item():.4f}")

### Atividades

##### 1 - Verificar se o modelo é bom

##### 2 - Como eu consigo melhorar esse modelo

##### 3 - Verificar o que o modelo aprendeu (o que tem dentro de nn.Linear)

##### 4 - Classificar em 3 grupos (X, Y ou Z) ao inves da probabilidade de ser do grupo X

#### 1 - Verificar se o modelo é bom
#### Verificar o conteudo da Acccuracy

In [5]:
modelo.eval()  # modo avaliação

# Desabilita o cálculo de gradientes
with torch.no_grad():
    # 1. Calcula as saídas do modelo
    outputs = modelo(X)
    # 2. Aplica sigmoid e converte para 0 ou 1
    # A função sigmoid retorna valores entre 0 e 1, então usamos um limiar de 0.5 para classificar
    predicted = (torch.sigmoid(outputs) > 0.5).float()
    # 3. Calcula a acurácia
    accuracy = (predicted == y).float().mean()
    print(f"Acurácia: {accuracy.item() * 100:.2f}%")

Acurácia: 70.67%


### 2 - Como eu posso melhorar o modelo?

#### Dividindo o dataset e um dataset de Treino e um de Teste (70/30) ou (80/20)

In [7]:
# Dividir dados em treino e teste
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

# Treinar modelo com dados de treino
modelo = torch.nn.Linear(4, 1)
funcao_perda = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(modelo.parameters(), lr=0.1)
for epoch in range(1000):
    optimizer.zero_grad()
    outputs = modelo(X_train)
    loss = funcao_perda(outputs, y_train)
    loss.backward()
    optimizer.step()

# Avaliar modelo com dados de teste
modelo.eval()
with torch.no_grad():
    outputs = modelo(X_test)
    predicted = (torch.sigmoid(outputs) > 0.5).float()
    accuracy = (predicted == y_test).float().mean()
    print(f"Acurácia no teste: {accuracy.item() * 100:.2f}%")

Acurácia no teste: 76.67%


### 3 - Verificar o que o modelo aprendeu (o que tem dentro de nn.Linear)

In [9]:
modelo.eval()  # modo avaliação

pesos = modelo.weight.data
print("Pesos do modelo:", pesos)

bias = modelo.bias.data
print("Bias do modelo:", bias)

Pesos do modelo: tensor([[ 0.7290, -2.0029,  0.3724, -1.2598]])
Bias do modelo: tensor([1.1494])


### 4 - Classificar em probabilidade de ser da classe X, Y ou Z

In [ ]:
modelo2 = torch.nn.Linear(4, 3)  # 4 features → 3 saídas (probabilidades para cada classe)

X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, iris.target, test_size=0.2, random_state=42)

y_multiclass = torch.tensor(y_train, dtype=torch.long)  # rótulos como inteiros para CrossEntropyLoss

#print("Rótulos Multiclasse:", y_multiclass)

perda = torch.nn.CrossEntropyLoss()  # combina softmax + CE
optimizer2 = torch.optim.SGD(modelo2.parameters(), lr=0.1)
for epoch in range(1000):
    optimizer2.zero_grad()
    outputs = modelo2(X_train)
    loss = perda(outputs, y_multiclass)  # CrossEntropyLoss espera rótulos inteiros
    loss.backward()
    optimizer2.step()

modelo2.eval()

with torch.no_grad():
    outputs = modelo2(X_train)
    predicted = torch.argmax(outputs, dim=1)  # classe com maior probabilidade
    accuracy = (predicted == y_multiclass).float().mean()
    print(f"Acurácia Multiclasse: {accuracy.item() * 100:.2f}%")

